# 02b — Segmentation des clients par K-means

**Objectif** : identifier des **profils comportementaux** de clients, puis mesurer le churn de chaque profil.

**Règle** : la variable `churn` **n'est pas utilisée** pour construire les clusters. Elle n'est lue qu'à la section 5, une fois les clusters nommés.

**Données** : `train.parquet` uniquement (80 000 clients). Pipeline dans `churn.segmentation.kmeans` : sélection des variables → imputation médiane → log sur les variables asymétriques → standardisation → KMeans (`random_state` = 42, `n_init` = 10).

In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display
from sklearn.decomposition import PCA

from churn import eda
from churn.charts import (correlation_heatmap, diverging_heatmap, k_selection_chart, pca_facets,
                          publish, rate_chart, rate_heatmap)
from churn.config import get_config
from churn.data.split import load_train
from churn.logging_setup import setup_logging
from churn.segmentation import kmeans as km

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

cfg = get_config()
train = load_train()
features_only = train.drop(columns=[cfg.data.target])   # la cible est retirée dès le départ
print(train.shape, "| variables de segmentation :", len(km.FEATURES))

(80000, 104) | variables de segmentation : 11


## 1. Variables de segmentation

| Bloc | Variables | Transformation |
|---|---|---|
| Usage et facture | `mou_Mean`, `rev_Mean`, `totmrc_Mean`, `ovrrev_Mean` | log(1 + x) |
| Tendance | `change_mou` | log symétrique : signe(x) · log(1 + \|x\|) |
| Ancienneté et terminal | `months`, `eqpdays`, `hnd_price` | aucune (distributions peu asymétriques) |
| Service client, réseau, lignes | `custcare_Mean`, `drop_blk_Mean`, `uniqsubs` | log(1 + x) |

**Pourquoi ces 11 variables** : une par dimension de comportement identifiée en E4 (volume, facture, dépassements, tendance, ancienneté, terminal, service client, qualité réseau, taille du compte), en évitant les doublons connus (`avg3mou`, `adjrev`, `ovrmou_Mean`…). **Pourquoi le log** : K-means minimise des distances euclidiennes ; sans transformation, quelques très gros consommateurs dicteraient les centres. **Pourquoi standardiser** : sinon `eqpdays` (en centaines de jours) écraserait `uniqsubs` (1 à 3).

In [2]:
X = km.transform_features(features_only)
shape = pd.DataFrame({"asymetrie_brute": train[km.FEATURES].skew(), "asymetrie_transformee": X.skew()}).round(2)
shape

,asymetrie_brute,asymetrie_transformee
mou_Mean,2.34,-1.54
rev_Mean,10.30,-0.16
totmrc_Mean,1.66,-1.53
ovrrev_Mean,6.46,0.67
custcare_Mean,34.91,1.53
drop_blk_Mean,5.67,0.05
uniqsubs,2.54,1.33
change_mou,17.02,0.18
months,1.06,1.06
eqpdays,1.05,1.05


In [3]:
corr = X.corr().round(2)
publish(correlation_heatmap(corr, "Aucune paire de variables de segmentation ne dépasse |r| = 0,75"),
        "02b_correlations", width=900)
display(eda.top_correlated_pairs(corr, threshold=0.5))

,var_1,var_2,rho
0,rev_Mean,totmrc_Mean,0.75
1,mou_Mean,drop_blk_Mean,0.66
2,mou_Mean,rev_Mean,0.65
3,rev_Mean,ovrrev_Mean,0.60
4,mou_Mean,totmrc_Mean,0.51
5,mou_Mean,ovrrev_Mean,0.51
6,rev_Mean,drop_blk_Mean,0.51


**Conclusion** : après transformation, les asymétries passent de 2 à 35 à moins de 1,6 en valeur absolue (`custcare_Mean` : 34,9 → 1,5 ; `change_mou` : 17,0 → 0,2). La plus forte corrélation entre variables retenues est **r = 0,75** (`rev_Mean` / `totmrc_Mean`) : aucune paire ne dépasse le seuil de 0,9, **les 11 variables sont conservées**. Le bloc usage-facture (`mou_Mean`, `rev_Mean`, `totmrc_Mean`, `ovrrev_Mean`, `drop_blk_Mean`, r de 0,5 à 0,75) pèsera tout de même lourd dans les distances : c'est un point à garder en tête pour lire les clusters.

## 2. Choix de k (2 à 10)

Inertie (coude), silhouette sur un échantillon de 10 000 clients, indice de Davies-Bouldin, et stabilité : ARI (*Adjusted Rand Index*) entre les partitions obtenues avec 5 graines différentes (1 = partitions identiques).

In [4]:
metrics = km.evaluate_k(X, range(2, 11))
metrics.round(3)

01:44:49 | INFO    | churn.segmentation.kmeans | k=2 évalué


01:45:04 | INFO    | churn.segmentation.kmeans | k=3 évalué


01:45:18 | INFO    | churn.segmentation.kmeans | k=4 évalué


01:45:36 | INFO    | churn.segmentation.kmeans | k=5 évalué


01:45:54 | INFO    | churn.segmentation.kmeans | k=6 évalué


01:46:18 | INFO    | churn.segmentation.kmeans | k=7 évalué


01:46:43 | INFO    | churn.segmentation.kmeans | k=8 évalué


01:47:09 | INFO    | churn.segmentation.kmeans | k=9 évalué


01:47:41 | INFO    | churn.segmentation.kmeans | k=10 évalué


,inertie,silhouette,davies_bouldin,ari_moyen,ari_min,plus_petit_cluster_pct
k,,,,,,
2,696677.096,0.180,1.871,0.997,0.994,49.5
3,630009.652,0.137,2.037,0.953,0.915,24.8
4,579646.183,0.146,1.867,0.991,0.985,10.3
5,541704.297,0.144,1.947,0.998,0.996,8.9
6,516185.218,0.143,1.883,0.829,0.638,7.3
7,493581.083,0.139,1.878,0.793,0.549,6.2
8,473501.365,0.131,1.885,0.991,0.983,6.1
9,456785.955,0.126,1.838,0.992,0.987,6.0
10,443134.243,0.126,1.859,0.972,0.938,4.7


In [5]:
CHOSEN_K = km.N_CLUSTERS
publish(k_selection_chart(metrics, CHOSEN_K, "k = 5 : le plus stable des k interprétables, silhouette comparable à k = 4"),
        "02b_choix_k", width=1000)

**Choix : k = 5.**

| Critère | Lecture |
|---|---|
| Coude | Pas de coude net : l'inertie décroît régulièrement (signe d'une structure peu marquée) |
| Silhouette | Maximale à k = 2 (0,180), puis stable entre 0,137 et 0,146 pour k = 3 à 7 ; k = 5 : **0,144** |
| Davies-Bouldin | k = 4 est meilleur (1,867) que k = 5 (1,947) ; l'écart est faible |
| Stabilité (ARI) | k = 5 est **le plus stable** (ARI min 0,996) ; k = 6 et 7 sont instables (0,64 et 0,55) |

k = 2 est écarté : il ne sépare que petits et gros consommateurs, trop pauvre pour un usage métier. Entre k = 4 et k = 5, les critères géométriques sont proches ; k = 5 est retenu pour sa **stabilité** et son **interprétabilité** : il sépare les gros consommateurs en deux groupes selon leur **tendance d'usage** (en baisse ou en hausse), distinction absente à k = 4 et utile pour détecter les départs.

**Qualité réelle de la séparation : faible.** Une silhouette de 0,144 (sur une échelle de −1 à 1) indique des clusters qui se chevauchent : les clients forment un continuum, et la segmentation est une **partition utile** plutôt que la découverte de groupes naturels.

## 3. Profilage des clusters (sans la cible)

In [6]:
bundle = km.fit_segmentation(features_only, CHOSEN_K)
pipeline, names = bundle["pipeline"], bundle["segment_names"]
labels = pipeline.predict(features_only)
Xs = pipeline[:-1].transform(features_only)
profile = km.cluster_profiles(features_only, labels)
print(f"silhouette (échantillon de 10 000) : {bundle['silhouette']:.3f}")
profile.round(1)

silhouette (échantillon de 10 000) : 0.144


,n,part_%,mou_Mean,rev_Mean,totmrc_Mean,ovrrev_Mean,custcare_Mean,drop_blk_Mean,uniqsubs,change_mou,months,eqpdays,hnd_price
cluster,,,,,,,,,,,,,
0,7118,8.9,92.5,14.1,10.0,0.0,0.0,1.3,2.0,-0.8,14.0,359.0,80.0
1,16890,21.1,769.2,74.4,52.5,15.9,1.7,14.0,1.0,-180.2,15.0,262.0,130.0
2,24362,30.5,220.5,38.8,45.0,0.0,0.0,2.7,1.0,-8.5,13.0,322.0,130.0
3,15515,19.4,781.2,74.0,54.2,15.9,1.3,12.7,1.0,167.5,15.0,247.0,130.0
4,16115,20.1,164.8,35.7,30.0,0.0,0.0,3.0,1.0,-5.8,27.0,723.0,30.0


In [7]:
z = km.cluster_zscores(Xs, labels).rename(index=names)
publish(diverging_heatmap(z, "Deux axes structurent les profils : le volume d'usage et l'ancienneté du couple client-terminal",
                          "z-score moyen", limit=2.2), "02b_zscores", width=1100)

**Lecture** : deux axes structurent les profils.

- **Volume d'usage et de facture** : deux clusters au-dessus de la moyenne sur toutes les variables d'usage (z ≈ +0,5 à +0,8), un cluster très en dessous (forfait −2,2 écarts-types).
- **Ancienneté du couple client-terminal** : un cluster se distingue par une ancienneté (+1,0) et un âge de terminal (+1,4) élevés, et un terminal bon marché (−1,0).

La tendance d'usage (`change_mou`) sépare nettement les deux clusters de gros consommateurs (−1,05 contre +1,23). `uniqsubs` n'est élevé que dans le cluster à petit forfait (+0,94 ; médiane de 2 lignes).

## 4. Noms métier

Les noms sont attribués par des **règles sur les profils** (`km.name_clusters`), pas par numéro de cluster : les numéros de KMeans sont arbitraires.

In [8]:
naming = profile.rename(index=names)[["n", "part_%", "mou_Mean", "totmrc_Mean", "change_mou",
                                              "months", "eqpdays", "hnd_price", "uniqsubs"]]
naming

,n,part_%,mou_Mean,totmrc_Mean,change_mou,months,eqpdays,hnd_price,uniqsubs
cluster,,,,,,,,,
Lignes secondaires à petit forfait,7118,8.9,92.50,10.00000,-0.75,14.0,359.0,79.98999,2.0
Gros consommateurs en baisse d'usage,16890,21.1,769.25,52.53000,-180.25,15.0,262.0,129.98999,1.0
"Usage modéré, clients récents",24362,30.5,220.50,44.99000,-8.50,13.0,322.0,129.98999,1.0
Gros consommateurs en hausse d'usage,15515,19.4,781.25,54.21625,167.50,15.0,247.0,129.98999,1.0
Clients anciens à terminal ancien et bon marché,16115,20.1,164.75,30.00000,-5.75,27.0,723.0,29.98999,1.0


| Nom métier | Part | Justification (médianes) |
|---|---|---|
| **Lignes secondaires à petit forfait** | 8,9 % | forfait de 10 $, 93 min/mois, **2 lignes** sur le compte : probablement une ligne d'appoint ou familiale |
| **Gros consommateurs en baisse d'usage** | 21,1 % | 769 min/mois, forfait de 53 $, dépassements (16 $), **usage en baisse de 180 min** |
| **Gros consommateurs en hausse d'usage** | 19,4 % | profil identique (781 min, 54 $, 16 $ de dépassements) mais **usage en hausse de 168 min** |
| **Usage modéré, clients récents** | 30,5 % | 221 min/mois, forfait de 45 $, **13 mois** d'ancienneté, terminal de 322 jours |
| **Clients anciens à terminal ancien et bon marché** | 20,1 % | **27 mois** d'ancienneté, terminal de **723 jours** acheté **30 $**, 165 min/mois |

## 5. Churn par cluster (la cible est lue seulement maintenant)

In [9]:
segment = pd.Series([names[i] for i in labels], index=train.index, name="segment")
seg_rates = eda.churn_rate_table(train, segment, sort_by_rate=True)
display(seg_rates)
test = eda.chi2_cramers_v(segment, train[cfg.data.target])
print(f"χ² = {test['chi2']:.1f} (ddl = {test['ddl']}), p = {test['p_value']:.1e}, V de Cramér = {test['cramers_v']:.3f}")

,n,churners,taux,ic_bas,ic_haut,part
segment,,,,,,
Clients anciens à terminal ancien et bon marché,16115,9010,55.91,55.14,56.68,20.14
Lignes secondaires à petit forfait,7118,3887,54.61,53.45,55.76,8.90
Gros consommateurs en baisse d'usage,16890,8370,49.56,48.80,50.31,21.11
"Usage modéré, clients récents",24362,11341,46.55,45.93,47.18,30.45
Gros consommateurs en hausse d'usage,15515,7042,45.39,44.61,46.17,19.39


χ² = 528.7 (ddl = 4), p = 4.1e-113, V de Cramér = 0.081


In [10]:
overall = 100 * train[cfg.data.target].mean()
publish(rate_chart(seg_rates, "Le churn varie de 45 % à 56 % selon le profil : un écart réel mais modéré",
                   overall, horizontal=True), "02b_churn_par_cluster", width=1000)

**Conclusion** : le churn va de **45,4 %** (gros consommateurs en hausse) à **55,9 %** (clients anciens à terminal ancien), soit 10,5 points d'écart. Le lien est significatif (χ² = 528,7, p < 10⁻¹¹²) mais **faible** (V de Cramér = 0,081).

Lecture par profil (associations, pas causes) :

- **clients anciens à terminal ancien et bon marché** (55,9 %) et **lignes secondaires à petit forfait** (54,6 %) : les plus exposés, cohérent avec E4 (terminal vieux et bon marché, petit forfait) ;
- à profil de consommation identique, **l'usage en baisse** va avec 4,2 points de churn de plus que l'usage en hausse (49,6 % contre 45,4 %) ;
- **usage modéré, clients récents** (46,6 %) : sous la moyenne, mais c'est le cluster qui compte le plus de clients à 11-12 mois (21 %), voir ci-dessous.

In [11]:
tenure = eda.business_segments(train)["anciennete"]
rate, n = eda.two_way_rate(train, segment, tenure)
publish(rate_heatmap(rate, n, "Dans chaque profil, la fin d'engagement (11-12 mois) reste le facteur dominant",
                     overall, "ancienneté", "segment K-means"), "02b_cluster_x_anciennete", width=1000)

**Conclusion** : dans **chaque** cluster, les clients à 11-12 mois d'ancienneté churnent à 59-66 %, contre 29 à 45 % entre 6 et 10 mois. La fin d'engagement l'emporte sur le profil de consommation : K-means ne la capte pas, car `months` n'est qu'une variable sur 11 et son effet n'est pas monotone. Le cluster « clients anciens » ne contient que des clients de 13 mois et plus ; à ancienneté comparable (≥ 13 mois), c'est lui qui churne le plus (55,9 %, contre 46 à 55 % pour les autres).

## 6. Projection PCA en 2D

PCA ajustée sur les 80 000 clients standardisés, 5 000 points tirés au hasard pour l'affichage. Un panneau par cluster : le cluster en bleu, les autres clients en gris.

In [12]:
pca = PCA(n_components=2, random_state=cfg.random_state).fit(Xs)
sample = Xs.sample(5_000, random_state=cfg.random_state)
coords = pd.DataFrame(pca.transform(sample), index=sample.index, columns=["PC1", "PC2"])
explained = tuple(pca.explained_variance_ratio_)
print(f"variance expliquée : PC1 {explained[0]:.1%}, PC2 {explained[1]:.1%}, total {sum(explained):.1%}")
display(pd.DataFrame(pca.components_.T, index=Xs.columns, columns=["PC1", "PC2"]).round(2))
publish(pca_facets(coords, segment.loc[sample.index],
                   "Les clusters se chevauchent : un continuum plutôt que des groupes isolés", explained),
        "02b_pca", width=1100)

variance expliquée : PC1 33.2%, PC2 14.1%, total 47.3%


,PC1,PC2
mou_Mean,0.44,0.12
rev_Mean,0.43,0.25
totmrc_Mean,0.35,0.13
ovrrev_Mean,0.32,0.22
custcare_Mean,0.29,0.00
drop_blk_Mean,0.39,0.14
uniqsubs,-0.06,-0.08
change_mou,0.01,-0.04
months,-0.15,0.58
eqpdays,-0.28,0.53


**Conclusion** : les deux premières composantes expliquent **47 %** de la variance (PC1 33 %, PC2 14 %). PC1 est un axe de **volume** (usage, facture, dépassements, qualité réseau), PC2 un axe **ancienneté et terminal** (`months` +0,58, `eqpdays` +0,53, `hnd_price` −0,46). Les clusters occupent des zones distinctes mais **contiguës**, sans frontière vide : la projection confirme la silhouette faible. Les deux clusters de gros consommateurs se superposent dans ce plan, car ce qui les sépare (`change_mou`) pèse presque zéro sur PC1 et PC2.

## 7. Comparaison avec la segmentation par règles (E4)

**AUC en validation croisée** : chaque client reçoit comme score le taux de churn de son segment, calculé sur les 4 autres folds. C'est une mesure équitable de la capacité d'une segmentation à **ordonner** les clients par risque, quel que soit son nombre de segments.

In [13]:
rules = eda.rule_segment_labels(eda.business_segments(train))
y = train[cfg.data.target]
comparison = []
for name, seg in [("K-means (k = 5)", segment), ("Règles E4 (ancienneté × usage × terminal)", rules),
                  ("Ancienneté seule (3 classes)", tenure.astype(str))]:
    table = eda.churn_rate_table(train, seg)
    big = table[table["n"] >= 200]
    auc, auc_std = km.segment_auc_cv(seg, y)
    comparison.append({
        "segmentation": name, "segments": seg.nunique(),
        "taux_min": big["taux"].min(), "taux_max": big["taux"].max(),
        "ecart_pts": round(big["taux"].max() - big["taux"].min(), 1),
        "cramers_v": round(eda.chi2_cramers_v(seg, y)["cramers_v"], 3),
        "auc_cv": round(auc, 3), "auc_cv_std": round(auc_std, 3),
    })
comparison = pd.DataFrame(comparison).set_index("segmentation")
comparison

,segments,taux_min,taux_max,ecart_pts,cramers_v,auc_cv,auc_cv_std
segmentation,,,,,,,
K-means (k = 5),5,45.39,55.91,10.5,0.081,0.544,0.005
Règles E4 (ancienneté × usage × terminal),22,29.49,76.64,47.2,0.210,0.617,0.004
Ancienneté seule (3 classes),3,33.78,63.48,29.7,0.173,0.582,0.004


**Laquelle sépare le mieux le churn ? Les règles, nettement.**

- Règles E4 : AUC en CV **0,617**, V = 0,210, taux de 29 % à 77 %.
- K-means : AUC **0,544**, V = 0,081, taux de 45 % à 56 %.
- Même l'**ancienneté seule** en 3 classes (AUC 0,582) fait mieux que K-means.

**Pourquoi** : K-means optimise la ressemblance des clients, pas le churn. Il est dominé par le volume d'usage (5 variables corrélées sur 11), faiblement lié au churn, et ne peut pas isoler le pic non monotone à 11-12 mois. Les règles, elles, ont été **construites à partir du churn** du train (E4) : leur avantage est en partie optimiste, même en CV, car les seuils ont été choisis sur ces mêmes données.

**Laquelle est la plus lisible ?** Les règles : trois critères explicites qu'un conseiller applique sans modèle. K-means donne 5 **profils** faciles à raconter, mais aux frontières floues (silhouette 0,144) et dont l'affectation exige le pipeline.

**Usage recommandé** : les **règles** pour cibler le risque, **K-means** pour **décrire la base clients** (personas marketing, page « Segments » du dashboard, adaptation du message de rétention au profil).

## 8. Sauvegarde du pipeline de segmentation

In [14]:
path = km.save_segmentation(bundle)
reloaded = km.load_segmentation(path)
check = km.assign_segments(features_only, reloaded)
print(path.name, "| k =", reloaded["k"], "| identique après rechargement :",
      bool((check["segment_id"].to_numpy() == labels).all()))
print("variables attendues :", reloaded["features"])
reloaded["segment_names"]

01:49:03 | INFO    | churn.segmentation.kmeans | Segmentation sauvegardée : C:\Users\malek\OneDrive\Desktop\telecom_churn\models\segmentation.joblib


segmentation.joblib | k = 5 | identique après rechargement : True
variables attendues : ['mou_Mean', 'rev_Mean', 'totmrc_Mean', 'ovrrev_Mean', 'custcare_Mean', 'drop_blk_Mean', 'uniqsubs', 'change_mou', 'months', 'eqpdays', 'hnd_price']


{0: 'Lignes secondaires à petit forfait',
 1: "Gros consommateurs en baisse d'usage",
 2: 'Usage modéré, clients récents',
 3: "Gros consommateurs en hausse d'usage",
 4: 'Clients anciens à terminal ancien et bon marché'}

Le fichier `models/segmentation.joblib` contient le pipeline entraîné (imputation, log, standardisation et KMeans appris sur le train), les noms métier, la liste des variables et la silhouette. `km.assign_segments(df)` attribue un segment à n'importe quel client : il servira dans les artefacts (E12) et sur la page « Segments » du dashboard.

## Conclusion

- **5 profils stables** (ARI ≥ 0,996) et nommables, mais **peu séparés** (silhouette 0,144) : une partition utile d'un continuum, pas des groupes naturels.
- **Churn par profil : de 45,4 % à 55,9 %**, lien faible (V = 0,081). La segmentation par règles E4 sépare beaucoup mieux le risque (AUC en CV 0,617 contre 0,544).
- **Le cluster n'est pas utilisé comme variable du modèle** (D44) : il apporte peu d'information prédictive au-delà des variables d'origine, qu'un modèle à arbres combine mieux lui-même.
- **Utilité** : décrire la base clients et contextualiser chaque client à risque (« gros consommateur en baisse d'usage » appelle une autre offre que « ligne secondaire à petit forfait »).